# Milestone 2 — Encoder Comparison: CLIP · BERT · GPT-2

**Optional extension:** *autoregressive text encoders (GPT-based embeddings).*

No generation, no new pipeline. The same 6,000 curated captions go through a third
encoder, and milestone 1's `sanity_checks.sibling_gap` reads off a third row.

**The question this answers.** The two captions of one COCO image describe the same scene, so an
encoder that represents *meaning* should place them closer together than two unrelated captions. The
gap between those two cosines is the measurement. A large sibling cosine alongside an equally large
random cosine is the failure mode to watch for: it means everything has been mapped into one narrow
cone, so the rankings carry almost no information.

**The point for the write-up.** It is tempting to say CLIP beats GPT-2 because GPT-2 is causally
masked. That is wrong — **CLIP's text tower is causally masked too** (verified below, straight from
the loaded model). What separates them is the training objective: contrastive image–text matching
against next-token prediction. One was trained to make a sentence vector that lines up with a picture;
the other was trained to guess the next word.

In [ ]:
# --- make src/ importable --------------------------------------------------
import sys
from pathlib import Path

def _find_src():
    roots = [Path.cwd(), *Path.cwd().parents]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        roots += sorted(kaggle_input.iterdir())
    for root in roots:
        if (root / "src" / "config.py").is_file():
            return root / "src"
    raise RuntimeError("could not find src/config.py -- is this running inside the repo?")

SRC = _find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"modules imported from: {SRC}")

---
## 0. Paths on a hosted runtime

**Does nothing on a normal checkout**, so this notebook still runs unchanged locally.

It matters because the repository is **code only** — `data/` is git-ignored. Milestone 2 needs
milestone 1's `data/processed/` (manifest, embeddings and the 512px photo cache, ~390 MB), and a
`git clone` will not bring that with it.

There are two ways to supply it, and this cell handles both:

- **Attach it as a Kaggle Dataset.** The cell finds whichever attached dataset holds
  `processed/pairs.parquet`, copies it to `/kaggle/working/data` and points `config.py` there. The
  copy is not optional — `/kaggle/input` is read-only and `cfg.ensure_dirs()` creates directories
  under `DATA_DIR`.
- **Clone the repo somewhere writable and run milestone 1 first.** Then nothing needs redirecting
  and this cell stays out of the way.

On Colab `/content` is writable, so the second case applies and this cell is a no-op there.

It must stay **above the import cell** — `config.py` resolves every path once, when it is first
imported, and setting the variables afterwards has no effect.

In [ ]:
# --- paths on a hosted runtime (no-op on a normal checkout) ----------------
import os
import shutil
from pathlib import Path

PROJECT_ROOT = SRC.parent


def _writable(path):
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".t2i_write_probe"
        probe.touch()
        probe.unlink()
        return True
    except OSError:
        return False


if Path("/kaggle/input").is_dir():
    attached = sorted(p for p in Path("/kaggle/input").iterdir() if p.is_dir())
    source = next((d for d in attached if (d / "processed" / "pairs.parquet").is_file()), None)

    if source is not None:
        work = Path("/kaggle/working/data")
        if not (work / "processed").is_dir():
            print(f"copying {source.name}/processed -> {work} ...")
            shutil.copytree(source / "processed", work / "processed")
        os.environ["T2I_DATA_DIR"] = str(work)
        os.environ["T2I_OUTPUT_DIR"] = "/kaggle/working/outputs"
        print(f"Kaggle: milestone 1 data from dataset '{source.name}'")
    elif _writable(PROJECT_ROOT / "data"):
        print(f"Kaggle: no milestone 1 dataset attached, but {PROJECT_ROOT} is writable --"
              " run milestone 1 in this session first to build data/processed/.")
    else:
        raise RuntimeError(
            f"{PROJECT_ROOT} is read-only and no attached dataset holds "
            "processed/pairs.parquet. Either attach milestone 1's data/ directory, or clone "
            "the repo into /kaggle/working and run milestone 1 first. Saw: "
            f"{[d.name for d in attached] or 'nothing attached'}"
        )
else:
    print("Not on Kaggle -- using the repository's own data/ and outputs/ directories.")

In [ ]:
import time, warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()

import config as cfg
import utils
from modules import bert_embeddings, clip_embeddings, gpt2_embeddings, sanity_checks, sd_pipeline, storage

cfg.ensure_dirs()
utils.set_seed()

manifest = storage.read_manifest()
CAPTIONS = manifest["caption"].tolist()

first = manifest[manifest["cap_rank"] == 0].set_index("image_id")["pair_idx"]
second = manifest[manifest["cap_rank"] == 1].set_index("image_id")["pair_idx"]
shared = first.index.intersection(second.index)
PAIR_A, PAIR_B = first.loc[shared].to_numpy(), second.loc[shared].to_numpy()

print(f"captions      : {len(CAPTIONS):,}")
print(f"sibling pairs : {len(PAIR_A):,} images with two captions each")
print(f"device        : {cfg.DEVICE}")

## 1. Is CLIP's text tower really causal?

Asked of the loaded model rather than taken on trust. CLIP builds a causal attention mask inside its
text transformer exactly the way GPT-2 does; the evidence is in the module source and in the shape of
the mask the model constructs.

In [ ]:
from transformers.models.clip import modeling_clip
import inspect

source = inspect.getsource(modeling_clip)
for marker in ["_create_4d_causal_attention_mask", "causal_attention_mask", "is_causal"]:
    print(f"  {marker:<34}{'present' if marker in source else 'absent'} in transformers' CLIP text model")

print()
print("GPT-2 and CLIP both mask future tokens; only the training objective differs.")

## 2. Encode with all three

CLIP and BERT are reused from milestone 1's cache when it is valid — they are the same captions and
the same models, so recomputing them would only prove that the cache works. GPT-2 is new and takes a
couple of minutes.

Each encoder is freed before the next loads. None of these is large, but the habit is what keeps the
generation notebooks inside 8 GB.

In [ ]:
timings = {}

clip_tok, clip_model, _ = clip_embeddings.load_joint()
print("CLIP  :", clip_embeddings.describe(clip_model))
t0 = time.time()
clip_joint = clip_embeddings.encode_text(CAPTIONS, clip_tok, clip_model)
timings["CLIP ViT-B/32 (pooled)"] = time.time() - t0
sd_pipeline.free(clip_model); del clip_model

In [ ]:
bert_tok, bert_model = bert_embeddings.load()
print("BERT  :", bert_embeddings.describe(bert_model))
t0 = time.time()
bert_cls, bert_mean = bert_embeddings.encode_text(CAPTIONS, bert_tok, bert_model)
timings["BERT (CLS / mean)"] = time.time() - t0
sd_pipeline.free(bert_model); del bert_model

In [ ]:
gpt2_tok, gpt2_model = gpt2_embeddings.load()
print("GPT-2 :", gpt2_embeddings.describe(gpt2_model))
print(f"        pad token set to {gpt2_tok.pad_token!r} (GPT-2 ships none)")
t0 = time.time()
gpt2_last, gpt2_mean = gpt2_embeddings.encode_text(CAPTIONS, gpt2_tok, gpt2_model)
timings["GPT-2 (last / mean)"] = time.time() - t0
sd_pipeline.free(gpt2_model); del gpt2_model

for name, array in [("gpt2_last", gpt2_last), ("gpt2_mean", gpt2_mean)]:
    print(f"saved {storage.save_embedding(name, array).name}  {array.shape}  {array.dtype}")

## 3. Sibling agreement

`sanity_checks.sibling_gap`, unchanged from milestone 1. Each encoder gets the same pairs and the same
shuffle, so the only variable is the encoder.

In [ ]:
ENCODERS = {
    "CLIP ViT-B/32 (pooled)": clip_joint,
    "BERT [CLS]": bert_cls,
    "BERT mean": bert_mean,
    "GPT-2 last token": gpt2_last,
    "GPT-2 mean": gpt2_mean,
}

scores = sanity_checks.sibling_gap(ENCODERS, PAIR_A, PAIR_B)
table = (pd.DataFrame(scores).T
         .rename(columns={"sibling_cos": "sibling", "random_cos": "random"})
         .sort_values("gap", ascending=False))
table["ratio"] = (table["sibling"] / table["random"]).round(3)
display(table.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
y = np.arange(len(table))
ax.barh(y - 0.2, table["sibling"], height=0.38, label="two captions of the same image")
ax.barh(y + 0.2, table["random"], height=0.38, label="two unrelated captions")
ax.set_yticks(y); ax.set_yticklabels(table.index); ax.invert_yaxis()
ax.set_xlabel("mean cosine similarity")
ax.set_title("Sibling agreement: does the encoder represent meaning?")
ax.legend(); ax.grid(axis="x", alpha=0.3)
for i, gap in enumerate(table["gap"]):
    ax.annotate(f"gap {gap:.3f}", (max(table['sibling'].iloc[i], table['random'].iloc[i]) + 0.01, i),
                va="center", fontsize=8)
fig.tight_layout()
print(utils.save_fig(fig, "m2_encoder_comparison"))
plt.show()

## 4. What the numbers say

Read the **gap**, not the sibling cosine. A row with a high sibling cosine *and* a high random cosine
has collapsed its captions into a narrow cone — every sentence looks like every other, and the
ordering is nearly meaningless. The `ratio` column makes that visible: it is how many times more
similar a true sibling is than a random pair.

For GPT-2 specifically, compare its two rows against each other. Last-token pooling is the
theoretically right choice for a causal model — only the final position has attended to the whole
sentence — while mean pooling averages in early positions that have seen almost nothing. Which one
actually wins here is the finding, not the assumption.

In [ ]:
best = table.index[0]
summary = {
    "n_captions": len(CAPTIONS),
    "n_sibling_pairs": int(len(PAIR_A)),
    "encode_seconds": {k: round(v, 1) for k, v in timings.items()},
    "scores": {k: {m: round(float(x), 4) for m, x in v.items()} for k, v in scores.items()},
    "ranking_by_gap": table.index.tolist(),
    "best": best,
    "note": ("CLIP's text tower is causally masked too; the training objective, not the mask, "
             "is what separates it from GPT-2."),
}
print(utils.save_table("milestone_2_encoders", summary))
print()
print(f"best separation: {best}  (gap {table.loc[best, 'gap']:.4f}, "
      f"{table.loc[best, 'ratio']:.2f}x random)")